# Assignment 2: Leaf Species Recognition with Convolutional Neural Networks

Introduction to Machine Learning — starter notebook

Classify scanned tree leaves into **15 species** with a convolutional network.

* `leaves/train/` — 600 labelled images (`train_labels.csv`: `id,label`)
* `leaves/test/` — 525 unlabelled images; you submit `predictions.csv` (`id,label`) for these

This notebook is a **baseline**, not a good solution. It runs end to end and writes a valid
`predictions.csv`. Your job is to make it better and to explain, with evidence, *why* each change helps.

**Runtime:** Runtime → Change runtime type → **T4 GPU**.

## 1. Get the data
Upload `leaves.zip` (from Classroom) with the Files panel on the left, **or** put it in your Google Drive and set `ZIP_PATH` below.

In [ ]:
import os, zipfile
ZIP_PATH = "/content/leaves.zip"          # or e.g. "/content/drive/MyDrive/leaves.zip"
if ZIP_PATH.startswith("/content/drive"):
    from google.colab import drive; drive.mount("/content/drive")
if not os.path.isdir("/content/leaves"):
    zipfile.ZipFile(ZIP_PATH).extractall("/content")
DATA = "/content/leaves"
print(len(os.listdir(f"{DATA}/train")), "train images,", len(os.listdir(f"{DATA}/test")), "test images")

In [ ]:
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV)

SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED)

labels = pd.read_csv(f"{DATA}/train_labels.csv")
classes = sorted(labels.label.unique())
cid = {c: i for i, c in enumerate(classes)}
test_ids = sorted(f[:-4] for f in os.listdir(f"{DATA}/test") if f.endswith(".png"))
print(len(classes), "classes;", labels.label.value_counts().min(), "-", labels.label.value_counts().max(), "images per class")

## 2. Load images

In [ ]:
IMG_SIZE = 64

def load(folder, ids, size=IMG_SIZE):
    arr = [np.asarray(Image.open(f"{folder}/{i}.png").convert("RGB").resize((size, size)), dtype=np.float32) / 255.0
           for i in ids]
    return torch.from_numpy(np.stack(arr)).permute(0, 3, 1, 2)      # N x 3 x H x W, values in [0, 1]

X = load(f"{DATA}/train", labels.id)
y = torch.tensor([cid[c] for c in labels.label])
X_test = load(f"{DATA}/test", test_ids)

# hold out 20% of the training images for validation
perm = torch.randperm(len(X))
n_val = len(X) // 5
val_idx, tr_idx = perm[:n_val], perm[n_val:]
X_tr, y_tr, X_val, y_val = X[tr_idx], y[tr_idx], X[val_idx], y[val_idx]
print(X_tr.shape, X_val.shape, X_test.shape)

## 3. Model

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes=15, width=16):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, width, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(width, 2 * width, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(2 * width, 4 * width, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.head = nn.Linear(4 * width, n_classes)

    def forward(self, x):
        return self.head(F.adaptive_avg_pool2d(self.features(x), 1).flatten(1))

model = SmallCNN().to(DEV)
print(sum(p.numel() for p in model.parameters()), "parameters")

## 4. Train

In [ ]:
EPOCHS = 5
BATCH_SIZE = 256
LR = 0.05
opt = torch.optim.SGD(model.parameters(), lr=LR)

def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        pred = torch.cat([model(X[i:i + 256].to(DEV)).argmax(1).cpu() for i in range(0, len(X), 256)])
    return (pred == y).float().mean().item()

for epoch in range(EPOCHS):
    model.train()
    order = torch.randperm(len(X_tr))
    for i in range(0, len(X_tr), BATCH_SIZE):
        idx = order[i:i + BATCH_SIZE]
        loss = F.cross_entropy(model(X_tr[idx].to(DEV)), y_tr[idx].to(DEV))
        opt.zero_grad(); loss.backward(); opt.step()
    print(f"epoch {epoch + 1}: loss {loss.item():.3f}  train acc {accuracy(model, X_tr, y_tr):.3f}  val acc {accuracy(model, X_val, y_val):.3f}")

## 5. Predict and save
Download `predictions.csv` from the Files panel and submit it on Classroom with your notebook.

In [ ]:
model.eval()
with torch.no_grad():
    pred = torch.cat([model(X_test[i:i + 256].to(DEV)).argmax(1).cpu() for i in range(0, len(X_test), 256)])
pd.DataFrame({"id": test_ids, "label": [classes[k] for k in pred]}).to_csv("predictions.csv", index=False)
print(pd.read_csv("predictions.csv").head())